# 01 — Revisión de datos

Objetivo: determinar qué información de Volve está disponible y qué condiciones requieren revisión antes de explorar o modelar. No se modifican originales ni se aplica limpieza automática.

**Recorrido:** carga y contexto → copias y vistas simplificadas → validaciones por pozo y flujo → síntesis de hallazgos → próximos pasos en el cuaderno 02.

Los controles específicos permiten revisar la calidad de los datos en el contexto operacional; las decisiones de limpieza se documentan al final.


In [7]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw").is_dir(), "Abrir desde la raíz del proyecto o notebooks/."
ARCHIVO = ROOT / "data" / "raw" / "Volve production data.xlsx"

# Los identificadores se conservan como texto.
diario = pd.read_excel(ARCHIVO, sheet_name="Daily Production Data", dtype={
    "NPD_WELL_BORE_CODE": "string", "NPD_FIELD_CODE": "string", "NPD_FACILITY_CODE": "string"
})
diario["DATEPRD"] = pd.to_datetime(diario["DATEPRD"], errors="raise")
mensual = pd.read_excel(ARCHIVO, sheet_name="Monthly Production Data", skiprows=[1],
                        na_values=["NULL"], dtype={"NPDCode": "string"})


## Cómo leer los nombres de las variables

Las tablas y los gráficos muestran nombres en español. En el código se conservan los nombres originales para poder rastrear cada variable hasta el Excel. Los códigos de pozos y los códigos `OP` y `WI` se conservan como identificadores; el significado de estos últimos sigue pendiente de confirmación.

La tabla de equivalencias permite relacionar ambas versiones. Las unidades y las interpretaciones pendientes no se dan por confirmadas al traducir.


In [8]:
# Traducciones de presentación: los datos conservan sus nombres y valores originales.
etiquetas_es = {'DATEPRD': 'Fecha del registro',
 'WELL_BORE_CODE': 'Identificador textual del pozo',
 'NPD_WELL_BORE_CODE': 'Código NPD del pozo',
 'NPD_WELL_BORE_NAME': 'Nombre del pozo',
 'NPD_FIELD_CODE': 'Código del campo',
 'NPD_FIELD_NAME': 'Nombre del campo',
 'NPD_FACILITY_CODE': 'Código de instalación',
 'NPD_FACILITY_NAME': 'Nombre de instalación',
 'ON_STREAM_HRS': 'Horas operativas (h)',
 'AVG_DOWNHOLE_PRESSURE': 'Presión promedio de fondo',
 'AVG_DOWNHOLE_TEMPERATURE': 'Temperatura promedio de fondo',
 'AVG_DP_TUBING': 'Diferencia de presión promedio en la tubería',
 'AVG_ANNULUS_PRESS': 'Presión promedio del espacio anular',
 'AVG_CHOKE_SIZE_P': 'Apertura promedio de la válvula de restricción',
 'AVG_CHOKE_UOM': 'Unidad de apertura de la válvula',
 'AVG_WHP_P': 'Presión promedio en cabeza de pozo',
 'AVG_WHT_P': 'Temperatura promedio en cabeza de pozo',
 'DP_CHOKE_SIZE': 'Diferencia de presión en la válvula de restricción',
 'BORE_OIL_VOL': 'Volumen de petróleo (unidad pendiente)',
 'BORE_GAS_VOL': 'Volumen de gas (unidad pendiente)',
 'BORE_WAT_VOL': 'Volumen de agua producida (unidad pendiente)',
 'BORE_WI_VOL': 'Volumen de agua inyectada (interpretación y unidad pendientes)',
 'FLOW_KIND': 'Tipo de flujo',
 'WELL_TYPE': 'Tipo de pozo (código)',
 'Wellbore name': 'Nombre del pozo',
 'NPDCode': 'Código NPD del pozo',
 'Year': 'Año',
 'Month': 'Mes',
 'On Stream': 'Horas operativas del mes (h)',
 'Oil': 'Petróleo mensual (Sm³)',
 'Gas': 'Gas mensual (Sm³)',
 'Water': 'Agua producida mensual (Sm³)',
 'GI': 'Gas inyectado mensual (interpretación pendiente; Sm³)',
 'WI': 'Agua inyectada mensual (interpretación pendiente; Sm³)',
 'tipo': 'Tipo de dato',
 'valores_distintos': 'Valores distintos',
 'faltantes_total_pct': 'Faltantes en el total (%)',
 'faltantes_produccion_pct': 'Faltantes en producción (%)',
 'faltantes_inyeccion_pct': 'Faltantes en inyección (%)',
 'decision_vista': 'Decisión para la vista',
 'motivo': 'Motivo',
 'columna': 'Variable',
 'horas_cero': 'Registros con 0 horas operativas',
 'horas_fuera_0_24': 'Horas operativas fuera de 0 a 24',
 'petroleo_con_horas_cero': 'Petróleo positivo con 0 horas operativas',
 'production_con_tipo_WI': 'Flujo de producción con tipo de pozo WI',
 'registros': 'Registros',
 'count': 'Cantidad de datos disponibles',
 'mean': 'Promedio',
 'std': 'Desviación estándar',
 'min': 'Mínimo',
 '25%': 'Percentil 25',
 '50%': 'Mediana (percentil 50)',
 '75%': 'Percentil 75',
 'max': 'Máximo'}
tipos_es = {
    "object": "Texto o datos mixtos", "string": "Texto", "str": "Texto",
    "int64": "Número entero", "float64": "Número decimal",
    "datetime64[ns]": "Fecha", "datetime64[us]": "Fecha", "bool": "Verdadero o falso",
}

def tabla_en_espanol(tabla):
    """Crear una copia para mostrar, sin cambiar el dataframe de análisis."""
    vista = tabla.copy()
    vista = vista.rename(columns=etiquetas_es, index=etiquetas_es)
    if vista.index.name is not None:
        vista.index.name = etiquetas_es.get(vista.index.name, vista.index.name)
    if vista.columns.name is not None:
        vista.columns.name = etiquetas_es.get(vista.columns.name, vista.columns.name)
    if "Tipo de flujo" in vista.columns:
        vista["Tipo de flujo"] = vista["Tipo de flujo"].replace({
            "production": "Producción", "injection": "Inyección"
        })
    if "Tipo de dato" in vista.columns:
        vista["Tipo de dato"] = vista["Tipo de dato"].replace(tipos_es)
    return vista


In [9]:
equivalencias = pd.DataFrame({
    "Variable en español": [etiquetas_es[c] for c in diario.columns],
    "Nombre original en Excel": diario.columns,
})
display(equivalencias)
print("Hoja mensual — equivalencias:")
display(pd.DataFrame({
    "Variable en español": [etiquetas_es[c] for c in mensual.columns],
    "Nombre original en Excel": mensual.columns,
}))


,Variable en español,Nombre original en Excel
0,Fecha del registro,DATEPRD
1,Identificador textual del pozo,WELL_BORE_CODE
2,Código NPD del pozo,NPD_WELL_BORE_CODE
3,Nombre del pozo,NPD_WELL_BORE_NAME
4,Código del campo,NPD_FIELD_CODE
5,Nombre del campo,NPD_FIELD_NAME
6,Código de instalación,NPD_FACILITY_CODE
7,Nombre de instalación,NPD_FACILITY_NAME
8,Horas operativas (h),ON_STREAM_HRS
9,Presión promedio de fondo,AVG_DOWNHOLE_PRESSURE


Hoja mensual — equivalencias:


,Variable en español,Nombre original en Excel
0,Nombre del pozo,Wellbore name
1,Código NPD del pozo,NPDCode
2,Año,Year
3,Mes,Month
4,Horas operativas del mes (h),On Stream
5,Petróleo mensual (Sm³),Oil
6,Gas mensual (Sm³),Gas
7,Agua producida mensual (Sm³),Water
8,Gas inyectado mensual (interpretación pendient...,GI
9,Agua inyectada mensual (interpretación pendien...,WI


## Contenido y estructura

In [10]:
print(f"Diario: {diario.shape[0]:,} registros, {diario.shape[1]} columnas")
print(f"Mensual: {mensual.shape[0]:,} registros, {mensual.shape[1]} columnas")
print("Período diario:", diario["DATEPRD"].min(), "a", diario["DATEPRD"].max())
display(tabla_en_espanol(diario.head()))
display(tabla_en_espanol(mensual.head()))
display(tabla_en_espanol(diario.dtypes.astype(str).rename("tipo").to_frame()))

Diario: 15,634 registros, 24 columnas
Mensual: 526 registros, 10 columnas
Período diario: 2007-09-01 00:00:00 a 2016-12-01 00:00:00


,Fecha del registro,Identificador textual del pozo,Código NPD del pozo,Nombre del pozo,Código del campo,Nombre del campo,Código de instalación,Nombre de instalación,Horas operativas (h),Presión promedio de fondo,...,Unidad de apertura de la válvula,Presión promedio en cabeza de pozo,Temperatura promedio en cabeza de pozo,Diferencia de presión en la válvula de restricción,Volumen de petróleo (unidad pendiente),Volumen de gas (unidad pendiente),Volumen de agua producida (unidad pendiente),Volumen de agua inyectada (interpretación y unidad pendientes),Tipo de flujo,Tipo de pozo (código)
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,Producción,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,Producción,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,Producción,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,Producción,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,Producción,OP


,Nombre del pozo,Código NPD del pozo,Año,Mes,Horas operativas del mes (h),Petróleo mensual (Sm³),Gas mensual (Sm³),Agua producida mensual (Sm³),Gas inyectado mensual (interpretación pendiente; Sm³),Agua inyectada mensual (interpretación pendiente; Sm³)
0,15/9-F-1 C,7405,2014,4,227.50000,11142.47,1597936.65,0.00,NaN,NaN
1,15/9-F-1 C,7405,2014,5,733.83334,24901.95,3496229.65,783.48,NaN,NaN
2,15/9-F-1 C,7405,2014,6,705.91666,19617.76,2886661.69,2068.48,NaN,NaN
3,15/9-F-1 C,7405,2014,7,742.41666,15085.68,2249365.75,6243.98,NaN,NaN
4,15/9-F-1 C,7405,2014,8,432.99166,6970.43,1048190.80,4529.75,NaN,NaN


,Tipo de dato
Fecha del registro,Fecha
Identificador textual del pozo,Texto
Código NPD del pozo,Texto
Nombre del pozo,Texto
Código del campo,Texto
Nombre del campo,Texto
Código de instalación,Texto
Nombre de instalación,Texto
Horas operativas (h),Número decimal
Presión promedio de fondo,Número decimal


## Diccionario original

Consultar el estado de verificación y las observaciones antes de asumir unidades.

In [ ]:
diccionario = pd.read_excel(ROOT / "data" / "raw" / "Diccionario_Volve.xlsx", sheet_name=None, header=3)
for hoja, tabla in diccionario.items():
    print(hoja)
    display(tabla.dropna(how="all"))

## Copias y vistas simplificadas para explorar

`diario` conserva las 24 columnas cargadas. Creamos `produccion` e `inyeccion` con `.copy()`, separando únicamente por `FLOW_KIND`, y luego dos vistas con menos columnas. Conservamos los índices originales para rastrear cada fila hasta `diario`; no son una clave del negocio.

La tabla siguiente muestra valores distintos (sin contar faltantes), disponibilidad global y por flujo, y la decisión para cada columna. Apartamos de las vistas sólo los metadatos de campo e instalación que efectivamente sean constantes y el identificador textual alternativo `WELL_BORE_CODE`. Conservamos el código y nombre NPD del pozo, `FLOW_KIND`, `WELL_TYPE`, las unidades y todas las mediciones, incluso si son constantes o tienen muchos faltantes. El identificador alternativo sigue disponible en las copias completas; no se usa aquí para unir tablas.

Estas vistas simplifican la lectura: no eliminan paradas ni atípicos, no imputan, no suavizan series y no son una selección de variables para modelos. Si aparece un flujo distinto o vacío, queda en `diario` y se informa para revisión. Las unidades pendientes siguen pendientes. No se exportan archivos nuevos.


In [11]:
# Copias completas por flujo: conservan todas las columnas y los índices originales.
produccion = diario.loc[diario["FLOW_KIND"].eq("production")].copy()
inyeccion = diario.loc[diario["FLOW_KIND"].eq("injection")].copy()

resumen_columnas = pd.DataFrame({
    "tipo": diario.dtypes.astype(str),
    "valores_distintos": diario.nunique(dropna=True),
    "faltantes_total_pct": diario.isna().mean() * 100,
    "faltantes_produccion_pct": produccion.isna().mean() * 100,
    "faltantes_inyeccion_pct": inyeccion.isna().mean() * 100,
}).round(2)
resumen_columnas.index.name = "columna"
resumen_columnas["decision_vista"] = "Conservar"
resumen_columnas["motivo"] = "Identificación, contexto o medición; revisar antes de excluir."

metadatos_contexto = [
    "NPD_FIELD_CODE", "NPD_FIELD_NAME", "NPD_FACILITY_CODE", "NPD_FACILITY_NAME"
]
# Sólo apartar estos metadatos si son constantes y no tienen faltantes.
columnas_constantes = [
    c for c in metadatos_contexto
    if resumen_columnas.loc[c, "valores_distintos"] == 1 and diario[c].notna().all()
]
columnas_apartadas = ["WELL_BORE_CODE"] + columnas_constantes
resumen_columnas.loc[columnas_apartadas, "decision_vista"] = "Apartar de la vista"
resumen_columnas.loc["WELL_BORE_CODE", "motivo"] = (
    "Identificador textual alternativo; se conservan código y nombre NPD del pozo."
)
resumen_columnas.loc[columnas_constantes, "motivo"] = (
    "Metadato constante sin faltantes; permanece en las copias completas."
)
display(tabla_en_espanol(resumen_columnas))

columnas_vista = [c for c in diario.columns if c not in columnas_apartadas]
produccion_vista = produccion.loc[:, columnas_vista].copy()
inyeccion_vista = inyeccion.loc[:, columnas_vista].copy()

print("Diario completo:", diario.shape)
print("Vista de producción:", produccion_vista.shape)
print("Vista de inyección:", inyeccion_vista.shape)
print("Columnas apartadas sólo de las vistas:", [etiquetas_es[c] for c in columnas_apartadas])
otros_flujos = ~diario["FLOW_KIND"].isin(["production", "injection"])
print("Registros con otro flujo o flujo faltante, conservados en diario:", otros_flujos.sum())
display(tabla_en_espanol(produccion_vista.head()))
display(tabla_en_espanol(inyeccion_vista.head()))


,Tipo de dato,Valores distintos,Faltantes en el total (%),Faltantes en producción (%),Faltantes en inyección (%),Decisión para la vista,Motivo
Variable,,,,,,,
Fecha del registro,Fecha,3327,0.00,0.00,0.00,Conservar,"Identificación, contexto o medición; revisar a..."
Identificador textual del pozo,Texto,7,0.00,0.00,0.00,Apartar de la vista,Identificador textual alternativo; se conserva...
Código NPD del pozo,Texto,7,0.00,0.00,0.00,Conservar,"Identificación, contexto o medición; revisar a..."
Nombre del pozo,Texto,7,0.00,0.00,0.00,Conservar,"Identificación, contexto o medición; revisar a..."
Código del campo,Texto,1,0.00,0.00,0.00,Apartar de la vista,Metadato constante sin faltantes; permanece en...
Nombre del campo,Texto,1,0.00,0.00,0.00,Apartar de la vista,Metadato constante sin faltantes; permanece en...
Código de instalación,Texto,1,0.00,0.00,0.00,Apartar de la vista,Metadato constante sin faltantes; permanece en...
Nombre de instalación,Texto,1,0.00,0.00,0.00,Apartar de la vista,Metadato constante sin faltantes; permanece en...
Horas operativas (h),Número decimal,925,1.82,0.00,4.40,Conservar,"Identificación, contexto o medición; revisar a..."


Diario completo: (15634, 24)
Vista de producción: (9161, 19)
Vista de inyección: (6473, 19)
Columnas apartadas sólo de las vistas: ['Identificador textual del pozo', 'Código del campo', 'Nombre del campo', 'Código de instalación', 'Nombre de instalación']
Registros con otro flujo o flujo faltante, conservados en diario: 0


,Fecha del registro,Código NPD del pozo,Nombre del pozo,Horas operativas (h),Presión promedio de fondo,Temperatura promedio de fondo,Diferencia de presión promedio en la tubería,Presión promedio del espacio anular,Apertura promedio de la válvula de restricción,Unidad de apertura de la válvula,Presión promedio en cabeza de pozo,Temperatura promedio en cabeza de pozo,Diferencia de presión en la válvula de restricción,Volumen de petróleo (unidad pendiente),Volumen de gas (unidad pendiente),Volumen de agua producida (unidad pendiente),Volumen de agua inyectada (interpretación y unidad pendientes),Tipo de flujo,Tipo de pozo (código)
0,2014-04-07,7405,15/9-F-1 C,0.0,0.00000,0.00000,0.00000,0.0,0.000000,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,Producción,WI
1,2014-04-08,7405,15/9-F-1 C,0.0,NaN,NaN,NaN,0.0,1.003059,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,Producción,OP
2,2014-04-09,7405,15/9-F-1 C,0.0,NaN,NaN,NaN,0.0,0.979008,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,Producción,OP
3,2014-04-10,7405,15/9-F-1 C,0.0,NaN,NaN,NaN,0.0,0.545759,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,Producción,OP
4,2014-04-11,7405,15/9-F-1 C,0.0,310.37614,96.87589,277.27826,0.0,1.215987,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,Producción,OP


,Fecha del registro,Código NPD del pozo,Nombre del pozo,Horas operativas (h),Presión promedio de fondo,Temperatura promedio de fondo,Diferencia de presión promedio en la tubería,Presión promedio del espacio anular,Apertura promedio de la válvula de restricción,Unidad de apertura de la válvula,Presión promedio en cabeza de pozo,Temperatura promedio en cabeza de pozo,Diferencia de presión en la válvula de restricción,Volumen de petróleo (unidad pendiente),Volumen de gas (unidad pendiente),Volumen de agua producida (unidad pendiente),Volumen de agua inyectada (interpretación y unidad pendientes),Tipo de flujo,Tipo de pozo (código)
9001,2007-09-01,5693,15/9-F-4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inyección,WI
9002,2007-09-02,5693,15/9-F-4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inyección,WI
9003,2007-09-03,5693,15/9-F-4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inyección,WI
9004,2007-09-04,5693,15/9-F-4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inyección,WI
9005,2007-09-05,5693,15/9-F-4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inyección,WI


## Mapa de los pozos y cobertura del calendario

Esta tabla describe todos los pozos, incluidos los de inyección. El período observado va desde el primer hasta el último registro de cada pozo: no demuestra que haya estado operando durante todo ese tiempo.

- **Días sin registro:** fechas ausentes dentro de ese período; no equivalen a producción cero.
- **Días con cero horas:** fechas con un registro que declara cero horas operativas; no confirman por sí solas la causa de una parada.
- **Días con horas faltantes:** fechas con al menos un registro sin ese dato.
- **Cobertura del calendario:** días con registro divididos por días del período observado.

Contamos fechas distintas para no confundir registros duplicados con días adicionales. Las categorías de horas podrían superponerse si hubiera varios registros por fecha. Producción e inyección se cuentan como registros y no se suman sus volúmenes.


In [12]:
filas_mapa = []
for codigo_pozo, datos_pozo in diario.groupby("NPD_WELL_BORE_CODE", sort=True):
    inicio = datos_pozo["DATEPRD"].min()
    fin = datos_pozo["DATEPRD"].max()
    dias_periodo = len(pd.date_range(inicio, fin, freq="D"))
    dias_registrados = datos_pozo["DATEPRD"].nunique()
    filas_mapa.append({
        "Código del pozo": codigo_pozo,
        "Pozo": " / ".join(sorted(datos_pozo["NPD_WELL_BORE_NAME"].dropna().unique())),
        "Primer registro": inicio,
        "Último registro": fin,
        "Registros totales": len(datos_pozo),
        "Registros de producción": datos_pozo["FLOW_KIND"].eq("production").sum(),
        "Registros de inyección": datos_pozo["FLOW_KIND"].eq("injection").sum(),
        "Otros flujos o sin dato": (~datos_pozo["FLOW_KIND"].isin(["production", "injection"])).sum(),
        "Días del período": dias_periodo,
        "Días con registro": dias_registrados,
        "Días sin registro": dias_periodo - dias_registrados,
        "Cobertura del calendario (%)": round(100 * dias_registrados / dias_periodo, 2),
        "Días con cero horas": datos_pozo.loc[datos_pozo["ON_STREAM_HRS"].eq(0), "DATEPRD"].nunique(),
        "Días con horas faltantes": datos_pozo.loc[datos_pozo["ON_STREAM_HRS"].isna(), "DATEPRD"].nunique(),
        "Duplicados de pozo y fecha": datos_pozo.duplicated(["DATEPRD"]).sum(),
    })
mapa_pozos = pd.DataFrame(filas_mapa).sort_values("Pozo").reset_index(drop=True)
display(mapa_pozos)

print("Relación entre flujo registrado y tipo de pozo (códigos originales):")
display(pd.crosstab(diario["FLOW_KIND"], diario["WELL_TYPE"]).rename(
    index={"production": "Producción", "injection": "Inyección"}
).rename_axis(index="Flujo registrado", columns="Tipo de pozo"))
print("Duplicados por pozo y fecha:", diario.duplicated(["NPD_WELL_BORE_CODE", "DATEPRD"]).sum())
# Mantener este resumen para la síntesis posterior del notebook.
huecos = mapa_pozos.set_index("Pozo")["Días sin registro"]


,Código del pozo,Pozo,Primer registro,Último registro,Registros totales,Registros de producción,Registros de inyección,Otros flujos o sin dato,Días del período,Días con registro,Días sin registro,Cobertura del calendario (%),Días con cero horas,Días con horas faltantes,Duplicados de pozo y fecha
0,7405,15/9-F-1 C,2014-04-07,2016-04-21,746,746,0,0,746,746,0,100.00,308,0,0
1,7078,15/9-F-11,2013-07-08,2016-09-17,1165,1165,0,0,1168,1165,3,99.74,43,0,0
2,5599,15/9-F-12,2008-02-12,2016-09-17,3056,3056,0,0,3141,3056,85,97.29,218,0,0
3,5351,15/9-F-14,2008-02-12,2016-09-17,3056,3056,0,0,3141,3056,85,97.29,332,0,0
4,7289,15/9-F-15 D,2014-01-12,2016-09-17,978,978,0,0,980,978,2,99.80,209,0,0
5,5693,15/9-F-4,2007-09-01,2016-12-01,3327,0,3327,0,3380,3327,53,98.43,345,152,0
6,5769,15/9-F-5,2007-09-01,2016-09-18,3306,160,3146,0,3306,3306,0,100.00,497,133,0


Relación entre flujo registrado y tipo de pozo (códigos originales):


Tipo de pozo,OP,WI
Flujo registrado,,
Inyección,0,6473
Producción,9143,18


Duplicados por pozo y fecha: 0


## Faltantes por flujo y pozo

Comparar la disponibilidad global y en producción, y profundizar por pozo para distinguir faltantes asociados al flujo de sensores con poca cobertura. Los porcentajes globales no bastan para decidir una imputación.


In [13]:
produccion = diario.loc[diario["FLOW_KIND"].eq("production")].copy()
display(tabla_en_espanol(pd.DataFrame({
    "faltantes_total_pct": diario.isna().mean() * 100,
    "faltantes_produccion_pct": produccion.isna().mean() * 100
}).round(2)))
variables = ["AVG_DOWNHOLE_PRESSURE", "AVG_ANNULUS_PRESS", "AVG_CHOKE_SIZE_P", "BORE_OIL_VOL"]
display(tabla_en_espanol(produccion.groupby("NPD_WELL_BORE_NAME")[variables].agg(lambda s: s.isna().mean() * 100).round(2)))

,Faltantes en el total (%),Faltantes en producción (%)
Fecha del registro,0.00,0.00
Identificador textual del pozo,0.00,0.00
Código NPD del pozo,0.00,0.00
Nombre del pozo,0.00,0.00
Código del campo,0.00,0.00
Nombre del campo,0.00,0.00
Código de instalación,0.00,0.00
Nombre de instalación,0.00,0.00
Horas operativas (h),1.82,0.00
Presión promedio de fondo,42.56,1.98


,Presión promedio de fondo,Presión promedio del espacio anular,Apertura promedio de la válvula de restricción,Volumen de petróleo (unidad pendiente)
Nombre del pozo,,,,
15/9-F-1 C,0.40,97.72,0.00,0.0
15/9-F-11,0.52,0.52,0.17,0.0
15/9-F-12,0.20,0.43,1.44,0.0
15/9-F-14,0.20,17.11,6.41,0.0
15/9-F-15 D,0.00,0.00,0.00,0.0
15/9-F-5,100.00,0.00,0.00,0.0


## Registros para revisar

Estas señales no prueban errores. Las horas cero pueden corresponder a paradas; los valores mayores a 24 requieren verificar el criterio de registro.

In [14]:
controles = pd.DataFrame({
    "horas_cero": diario["ON_STREAM_HRS"].eq(0),
    "horas_fuera_0_24": diario["ON_STREAM_HRS"].lt(0) | diario["ON_STREAM_HRS"].gt(24),
    "petroleo_con_horas_cero": diario["BORE_OIL_VOL"].gt(0) & diario["ON_STREAM_HRS"].eq(0),
    "production_con_tipo_WI": diario["FLOW_KIND"].eq("production") & diario["WELL_TYPE"].eq("WI")
})
display(tabla_en_espanol(controles.sum().rename("registros").to_frame()))
display(tabla_en_espanol(diario.loc[controles.drop(columns="horas_cero").any(axis=1)]))

,Registros
Registros con 0 horas operativas,1952
Horas operativas fuera de 0 a 24,20
Petróleo positivo con 0 horas operativas,1
Flujo de producción con tipo de pozo WI,18


,Fecha del registro,Identificador textual del pozo,Código NPD del pozo,Nombre del pozo,Código del campo,Nombre del campo,Código de instalación,Nombre de instalación,Horas operativas (h),Presión promedio de fondo,...,Unidad de apertura de la válvula,Presión promedio en cabeza de pozo,Temperatura promedio en cabeza de pozo,Diferencia de presión en la válvula de restricción,Volumen de petróleo (unidad pendiente),Volumen de gas (unidad pendiente),Volumen de agua producida (unidad pendiente),Volumen de agua inyectada (interpretación y unidad pendientes),Tipo de flujo,Tipo de pozo (código)
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.00000,0.000000,...,%,0.000000,0.000000,0.000000,0.00,0.00,0.00,NaN,Producción,WI
91,2014-07-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,24.00000,215.070057,...,%,52.663308,54.966429,24.233500,522.00,77846.76,154.47,NaN,Producción,WI
202,2014-10-26,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,25.00000,220.957970,...,%,36.554275,47.287813,7.917633,333.14,51384.55,336.05,NaN,Producción,OP
856,2013-10-27,NO 15/9-F-11 H,7078,15/9-F-11,3420717,VOLVE,369304,MÆRSK INSPIRER,24.30839,256.047878,...,%,82.213141,60.017753,53.861978,1219.94,188361.38,69.99,NaN,Producción,OP
1218,2014-10-26,NO 15/9-F-11 H,7078,15/9-F-11,3420717,VOLVE,369304,MÆRSK INSPIRER,25.00000,220.578730,...,%,52.942725,63.461084,24.513722,1110.58,171297.30,339.89,NaN,Producción,OP
1301,2015-01-17,NO 15/9-F-11 H,7078,15/9-F-11,3420717,VOLVE,369304,MÆRSK INSPIRER,0.00000,219.800000,...,%,50.400000,64.800000,21.000000,1026.57,150773.50,461.02,NaN,Producción,OP
2166,2008-10-26,NO 15/9-F-12 H,5599,15/9-F-12,3420717,VOLVE,369304,MÆRSK INSPIRER,25.00000,233.664638,...,%,70.283581,75.273871,37.889267,3836.99,547535.29,12.84,NaN,Producción,OP
2527,2009-10-25,NO 15/9-F-12 H,5599,15/9-F-12,3420717,VOLVE,369304,MÆRSK INSPIRER,24.83333,261.865161,...,%,77.557616,75.845229,45.036169,4190.37,585516.23,279.62,NaN,Producción,OP
2891,2010-10-31,NO 15/9-F-12 H,5599,15/9-F-12,3420717,VOLVE,369304,MÆRSK INSPIRER,25.00000,0.000000,...,%,52.641503,88.527274,20.099256,1677.68,248776.29,3676.04,NaN,Producción,OP
3912,2013-10-27,NO 15/9-F-12 H,5599,15/9-F-12,3420717,VOLVE,369304,MÆRSK INSPIRER,24.14167,0.000000,...,%,33.458990,89.321837,4.623115,280.98,43383.87,4798.50,NaN,Producción,OP


## Síntesis de la revisión

Esta tabla conecta las señales revisadas con evidencia calculada y el paso siguiente. Se actualiza al ejecutar el cuaderno; no autoriza a limpiar datos automáticamente. Las unidades y correlaciones siguen pendientes de interpretación.


In [15]:
faltantes_anular = produccion.groupby("NPD_WELL_BORE_NAME")["AVG_ANNULUS_PRESS"].apply(
    lambda s: s.isna().mean() * 100
)
pozo_mayor_faltante = faltantes_anular.idxmax()
horas_produccion = produccion["ON_STREAM_HRS"]

sintesis = pd.DataFrame([
    {
        "Señal": "Faltantes de petróleo",
        "Evidencia": (
            f"Total: {diario['BORE_OIL_VOL'].isna().mean():.1%}; "
            f"producción: {produccion['BORE_OIL_VOL'].isna().mean():.1%}."
        ),
        "Interpretación / siguiente paso": "Interpretar por flujo; no completar con cero campos que pueden no aplicar."
    },
    {
        "Señal": "Disponibilidad de presión anular",
        "Evidencia": f"Mayor faltante en producción: {pozo_mayor_faltante}, {faltantes_anular.max():.1f}%.",
        "Interpretación / siguiente paso": "Revisar cobertura del sensor por período antes de seleccionar variables."
    },
    {
        "Señal": "Horas operativas",
        "Evidencia": (
            f"En producción: {horas_produccion.eq(0).sum()} registros con cero horas; "
            f"{(horas_produccion.lt(0) | horas_produccion.gt(24)).sum()} fuera de [0, 24]."
        ),
        "Interpretación / siguiente paso": "Ceros compatibles con paradas; verificar criterio de registro para valores fuera de rango."
    },
    {
        "Señal": "Petróleo y horas",
        "Evidencia": f"{controles['petroleo_con_horas_cero'].sum()} registros del diario con petróleo positivo y horas cero.",
        "Interpretación / siguiente paso": "Revisar asignación de volumen y fecha; no corregir sin evidencia."
    },
    {
        "Señal": "Clave y continuidad temporal",
        "Evidencia": (
            f"{diario.duplicated(['NPD_WELL_BORE_CODE', 'DATEPRD']).sum()} duplicados pozo-fecha; "
            f"{huecos.gt(0).sum()} pozos con días ausentes entre su inicio y fin."
        ),
        "Interpretación / siguiente paso": "Usar calendario explícito; no confundir el siguiente registro con el día siguiente."
    },
    {
        "Señal": "Coherencia de categorías",
        "Evidencia": f"{controles['production_con_tipo_WI'].sum()} registros combinan flujo de producción con el código de tipo de pozo WI.",
        "Interpretación / siguiente paso": "Revisar tipo de pozo y contexto antes de definir filtros."
    },
])
display(sintesis)


,Señal,Evidencia,Interpretación / siguiente paso
0,Faltantes de petróleo,Total: 41.4%; producción: 0.0%.,Interpretar por flujo; no completar con cero c...
1,Disponibilidad de presión anular,"Mayor faltante en producción: 15/9-F-1 C, 97.7%.",Revisar cobertura del sensor por período antes...
2,Horas operativas,En producción: 1141 registros con cero horas; ...,Ceros compatibles con paradas; verificar crite...
3,Petróleo y horas,1 registros del diario con petróleo positivo y...,Revisar asignación de volumen y fecha; no corr...
4,Clave y continuidad temporal,0 duplicados pozo-fecha; 5 pozos con días ause...,Usar calendario explícito; no confundir el sig...
5,Coherencia de categorías,18 registros combinan flujo de producción con ...,Revisar tipo de pozo y contexto antes de defin...


### Conclusión y continuidad

La revisión deja señales identificadas y controles reproducibles, pero no establece todavía un conjunto de datos apto para entrenamiento. No se han aplicado exclusiones ni imputaciones.

En `02_exploracion.ipynb`, revisar primero los pozos y períodos señalados: comparar producción, horas y disponibilidad de sensores. Contrastar allí las correlaciones globales antes de seleccionar variables. La reconciliación diaria/mensual y la confirmación de unidades siguen pendientes de incorporación o documentación dentro del flujo reproducible.

Registrar abajo las decisiones de limpieza propuestas, su evidencia y los pendientes. Una señal sin explicación permanece pendiente; no se transforma automáticamente en un error confirmado.


## Notas del análisis

- Unidades y definiciones pendientes:
- Inconsistencias que necesitan revisión:
- Variables utilizables por pozo:
- Criterios de limpieza propuestos y justificación:
